In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor


In [2]:
# 1. Load Data
train_df = pd.read_csv('../dataset/train.csv')
test_df = pd.read_csv('../dataset/test.csv')


In [5]:
train_df.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,is_train
0,0,qp02z1,48,0.0,0.048804,NaN,1,Not Allowed,No,NaN,NaN,1
1,1,qp02zt,48,0.0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,1
2,2,qp08bj,48,0.0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,1
3,3,qp08gt,48,0.0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy,1
4,4,qp02zq,48,0.0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,1


In [6]:
train_df.info(), test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77299 entries, 0 to 77298
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          77299 non-null  int64  
 1   geohash        77299 non-null  object 
 2   day            77299 non-null  int64  
 3   timestamp      77299 non-null  float64
 4   demand         77299 non-null  float64
 5   RoadType       76699 non-null  object 
 6   NumberofLanes  77299 non-null  int64  
 7   LargeVehicles  77299 non-null  object 
 8   Landmarks      77299 non-null  object 
 9   Temperature    74804 non-null  float64
 10  Weather        76502 non-null  object 
 11  is_train       77299 non-null  int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 7.1+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41778 entries, 0 to 41777
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index      

(None, None)

In [15]:
train_df['day'].unique(), test_df['day'].unique()

(array([48, 49]), array([49]))

In [7]:
# 2. Combine for consistent preprocessing
train_df['is_train'] = 1
test_df['is_train'] = 0
test_df['demand'] = np.nan # Target column placeholder for test data

full_df = pd.concat([train_df, test_df], ignore_index=True)

In [8]:
# 3. Clean Missing Values
full_df['Temperature'] = full_df['Temperature'].fillna(full_df['Temperature'].median())
full_df['RoadType'] = full_df['RoadType'].fillna('Unknown')
full_df['Weather'] = full_df['Weather'].fillna('Unknown')


In [10]:
full_df.isnull().sum()

Index                0
geohash              0
day                  0
timestamp            0
demand           41778
RoadType             0
NumberofLanes        0
LargeVehicles        0
Landmarks            0
Temperature          0
Weather              0
is_train             0
dtype: int64

### Feature Engineering

In [ ]:
#1 Time Features
full_df["timestamp"] = pd.to_datetime(full_df["timestamp"], format="%H:%M", errors='coerce')
# Fallback for train format which might just be floats or different format in your original data
full_df['timestamp'] = full_df['timestamp'].fillna(pd.to_datetime(train_df["timestamp"], errors='ignore'))


C:\Users\suraj rajak\AppData\Local\Temp\ipykernel_9172\651612943.py:4: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  full_df['timestamp'] = full_df['timestamp'].fillna(pd.to_datetime(train_df["timestamp"], errors='ignore'))


In [11]:
full_df['hour'] = full_df['timestamp'].dt.hour
full_df['minute'] = full_df['timestamp'].dt.minute
# Convert time into a single continuous continuous variable for easier trend tracking
full_df['time_in_minutes'] = (full_df['hour'] * 60) + full_df['minute']


In [12]:

#2 Day Features (Assuming 'day' represents sequential days)
# Modulo 7 extracts the day of the week (0-6)
full_df['day_of_week'] = full_df['day'] % 7
# Assume weekends are days 5 and 6 (Saturday/Sunday)
full_df['is_weekend'] = full_df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)


In [13]:
full_df.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,is_train,hour,minute,time_in_minutes,day_of_week,is_weekend
0,0,qp02z1,48,1970-01-01,0.048804,Unknown,1,Not Allowed,No,16.415162,Unknown,1,0,0,0,6,1
1,1,qp02zt,48,1970-01-01,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,1,0,0,0,6,1
2,2,qp08bj,48,1970-01-01,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,1,0,0,0,6,1
3,3,qp08gt,48,1970-01-01,0.003272,Residential,1,Not Allowed,No,16.415162,Rainy,1,0,0,0,6,1
4,4,qp02zq,48,1970-01-01,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,1,0,0,0,6,1


In [ ]:
# 3. Spatial Features (Geohash grouping)
# The first 4 characters of a geohash represent a larger bounding box (neighborhood)
full_df['geohash_zone'] = full_df['geohash'].astype(str).str[:4]

full_df.drop(columns=['timestamp'], inplace=True)


### Encoding

In [17]:
categorical_cols = ['geohash', 'geohash_zone', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
for col in categorical_cols:
    full_df[col] = full_df[col].astype('category')


### Train

In [18]:
# 6. Split back into Train and Test
train_data = full_df[full_df['is_train'] == 1].drop(columns=['is_train', 'Index'])
test_data = full_df[full_df['is_train'] == 0].drop(columns=['is_train', 'demand'])
test_indices = full_df[full_df['is_train'] == 0]['Index']


In [19]:
X = train_data.drop(columns=['demand'])
y = train_data['demand']

In [20]:
# Train/Validation Split
train_X, valid_X, train_y, valid_y = train_test_split(X, y, test_size=0.2, random_state=42)


In [21]:
model = XGBRegressor(
    n_estimators=1500,          
    learning_rate=0.03,         # Lower learning rate for better generalization
    max_depth=8,                # Reduced from 100 to prevent overfitting
    subsample=0.8,              # Uses 80% of data per tree (prevents overfitting)
    colsample_bytree=0.8,       # Uses 80% of features per tree (prevents overfitting)
    tree_method='hist',         
    enable_categorical=True,    
    early_stopping_rounds=100,  
    random_state=42
)

In [22]:
model.fit(
    train_X, train_y,
    eval_set=[(valid_X, valid_y)], 
    verbose=100                  
)


[0]	validation_0-rmse:0.13855
[100]	validation_0-rmse:0.04848
[200]	validation_0-rmse:0.04791
[248]	validation_0-rmse:0.04812


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,100
,enable_categorical,True
,eval_metric,None


### Evaluate

In [23]:
pred = model.predict(valid_X)
print(f"Improved R2 Score: {r2_score(valid_y, pred):.4f}")


Improved R2 Score: 0.8864


### Generate submission

In [24]:
test_pred = model.predict(test_data.drop(columns=['Index']))

submit_ans = pd.DataFrame({
    'Index': test_indices,
    'demand': test_pred
})

submit_ans.to_csv('b_submission_improved.csv', index=False)